<a href="https://colab.research.google.com/github/Biel07-max/Projeto-analise-de-dados/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install plotly openpyxl ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
import re
import io
from IPython.display import display, HTML, clear_output

In [ ]:

from google.colab import files

print("ENVIE SUA PLANILHA EXCEL:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Você precisa enviar uma planilha.")

nome = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[nome]))

df.columns = df.columns.astype(str).str.strip()
df = df.dropna(how="all").copy()

# Colunas reais da planilha
COL_TEC = "Tecnologias e Diretrizes para COVID - 19"
COL_DEM = "Demandante"
COL_INICIAL = "Análise inicial da Conitec"
COL_FINAL = "Análise Final da Conitec"
COL_DECISAO = "Decisão do Ministério da Saúde"

obrigatorias = [
    COL_TEC, COL_INICIAL, COL_FINAL, COL_DECISAO
]

faltantes = [c for c in obrigatorias if c not in df.columns]

if faltantes:
    raise ValueError(
        "Colunas não encontradas: " + str(faltantes)
        + "\nColunas disponíveis: " + str(df.columns.tolist())
    )


for c in df.columns:
    if df[c].dtype == "object":
        df[c] = df[c].astype("string").str.strip()

df[COL_TEC] = df[COL_TEC].fillna("Não informado")
df[COL_INICIAL] = df[COL_INICIAL].fillna("Não informado")
df[COL_FINAL] = df[COL_FINAL].fillna("Não informado")
df[COL_DECISAO] = df[COL_DECISAO].fillna("Não informado")

ENVIE SUA PLANILHA EXCEL:


Saving Tecnologias e Diretrizes para tratamento e prevenção da Covid-19_Página 1_Tabela.xlsx to Tecnologias e Diretrizes para tratamento e prevenção da Covid-19_Página 1_Tabela (4).xlsx


In [ ]:

class Visualizador:

    def __init__(self, dataframe, tema="plotly_white"):
        self.__df = dataframe.copy()
        self.tema = tema

    @property
    def dados(self):
        return self.__df.copy()

    def barras(self, coluna, titulo):
        dados = (
            self.__df[coluna]
            .value_counts()
            .reset_index()
        )
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Categoria",
            y="Quantidade",
            text="Quantidade",
            color="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            xaxis_title="Categorias",
            yaxis_title="Quantidade",
            showlegend=False
        )
        return fig

    def barras_horizontais(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().head(10).reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Quantidade",
            y="Categoria",
            orientation="h",
            text="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            yaxis={"categoryorder": "total ascending"}
        )
        return fig

    def pizza(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.pie(
            dados,
            names="Categoria",
            values="Quantidade",
            hole=0.4,
            title=titulo,
            template=self.tema
        )
        fig.update_traces(textinfo="percent+label")
        fig.update_layout(title_x=0.5)
        return fig

    def comparar(self):
        inicial = self.__df[COL_INICIAL].value_counts()
        final = self.__df[COL_FINAL].value_counts()

        categorias = sorted(set(inicial.index) | set(final.index))

        comparacao = pd.DataFrame({
            "Categoria": categorias,
            "Análise inicial": [
                int(inicial.get(c, 0)) for c in categorias
            ],
            "Análise final": [
                int(final.get(c, 0)) for c in categorias
            ]
        })

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise inicial"],
            name="Análise inicial"
        ))

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise final"],
            name="Análise final"
        ))

        fig.update_layout(
            title="Comparação entre análise inicial e final da Conitec",
            barmode="group",
            template=self.tema,
            title_x=0.5,
            xaxis_title="Resultado",
            yaxis_title="Quantidade"
        )

        return fig

    def treemap(self):
        dados = self.__df[COL_DECISAO].value_counts().reset_index()
        dados.columns = ["Decisão", "Quantidade"]

        return px.treemap(
            dados,
            path=["Decisão"],
            values="Quantidade",
            title="Mapa de distribuição das decisões",
            template=self.tema
        )

    def evolucao(self):
        if COL_DEM not in self.__df.columns:
            return None

        texto = self.__df[COL_DEM].astype(str)

        datas = texto.str.extract(
            r"(\d{1,2}/\d{1,2}/\d{4})"
        )[0]

        datas = pd.to_datetime(
            datas, dayfirst=True, errors="coerce"
        )

        anos = datas.dropna().dt.year.value_counts().sort_index()

        if anos.empty:
            return None

        dados = anos.reset_index()
        dados.columns = ["Ano", "Quantidade"]

        fig = px.line(
            dados,
            x="Ano",
            y="Quantidade",
            markers=True,
            title="Quantidade de registros por ano",
            template=self.tema
        )
        fig.update_layout(title_x=0.5)
        return fig

    def tabela(self):
        return self.__df


In [ ]:
class Dashboard:

    def __init__(self, dataframe):
        self.__df = dataframe.copy()

    def indicadores(self):

        total = len(self.__df)
        tecnologias = self.__df[COL_TEC].nunique()

        favoraveis = self.__df[COL_FINAL].str.contains(
            "favorável", case=False, na=False
        ).sum()

        incorporadas = self.__df[COL_DECISAO].str.contains(
            "incorporação", case=False, na=False
        ).sum()

        nao_incorporadas = self.__df[COL_DECISAO].str.contains(
            "não incorporação", case=False, na=False
        ).sum()

        cards = [
            ("Total de registros", total),
            ("Tecnologias analisadas", tecnologias),
            ("Análises finais favoráveis", int(favoraveis)),
            ("Incorporações e manutenções", int(incorporadas)),
            ("Não incorporações", int(nao_incorporadas))
        ]

        html = """
        <div style="display:flex;flex-wrap:wrap;gap:12px;">
        """

        for titulo, valor in cards:
            html += f"""
            <div style="
                flex:1;
                min-width:145px;
                background:#17365D;
                color:white;
                padding:20px;
                border-radius:12px;
                text-align:center;
                margin-bottom:10px;
            ">
                <div style="font-size:13px;">{titulo}</div>
                <div style="font-size:29px;font-weight:bold;">
                    {valor}
                </div>
            </div>
            """

        html += "</div>"
        display(HTML(html))

    def executar(self):

        if self.__df.empty:
            display(HTML("<h3>Nenhum registro encontrado.</h3>"))
            return

        viz = Visualizador(self.__df)

        display(HTML("""
        <div style="
        background:#17365D;
        color:white;
        padding:25px;
        border-radius:12px;
        ">
        <h1>PAINEL COVID-19</h1>
        <p>Tecnologias, tratamentos, prevenção e decisões da Conitec</p>
        <p>Projeto de Programação Orientada a Objetos</p>
        </div>
        """))

        # INDICADORES
        display(HTML("<h2>Indicadores principais</h2>"))
        self.indicadores()

        # GRÁFICO DE BARRAS
        display(HTML("<h2>1. Distribuição das decisões</h2>"))
        display(viz.barras(
            COL_DECISAO,
            "Decisões do Ministério da Saúde"
        ))

        # GRÁFICO DE PIZZA
        display(HTML("<h2>2. Participação das decisões</h2>"))
        display(viz.pizza(
            COL_DECISAO,
            "Percentual de cada tipo de decisão"
        ))

        # COMPARAÇÃO
        display(HTML("<h2>3. Comparação das análises</h2>"))
        display(viz.comparar())

        # TREEMAP
        display(HTML("<h2>4. Visualização adicional - Treemap</h2>"))
        display(viz.treemap())

        # DEMANDANTES
        if COL_DEM in self.__df.columns:
            display(HTML("<h2>5. Principais demandantes</h2>"))
            display(viz.barras_horizontais(
                COL_DEM,
                "Demandantes das tecnologias"
            ))

        # EVOLUÇÃO
        display(HTML("<h2>6. Evolução temporal</h2>"))
        grafico = viz.evolucao()

        if grafico is not None:
            display(grafico)
        else:
            display(HTML(
                "<p>Não foram identificadas datas suficientes "
                "para montar a evolução temporal.</p>"
            ))

        # TABELA
        display(HTML("<h2>7. Tabela completa de dados</h2>"))
        display(
            viz.tabela().reset_index(drop=True).style
            .set_properties(**{
                "text-align": "left",
                "white-space": "normal"
            })
            .set_table_styles([
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#17365D"),
                        ("color", "white"),
                        ("padding", "10px")
                    ]
                }
            ])
        )

In [ ]:
print("DADOS CARREGADOS:", len(df), "registros")
print("COLUNAS:", len(df.columns))

dashboard = Dashboard(df)
dashboard.executar()


DADOS CARREGADOS: 20 registros
COLUNAS: 16


,Qtd,Tecnologias e Diretrizes para COVID - 19,Demandante,Análise inicial da Conitec,Relatório de Recomendação inicial,Relatórios para a Sociedade,Consulta Pública,Contribuições de experiência e opinião,Contribuições técnico - científicas,Análise Final da Conitec,Relatório de recomendação Final,Decisão do Ministério da Saúde,Decisão do Ministério da Saúde - Relatório,Decisão do Ministério da Saúde - Portarias,Notas Técnicas,Despacho
0,15,"Nirmatrelvir/Ritonavir para o tratamento da Covid-19 para pacientes com sintomas leves a moderados, que não requerem oxigênio suplementar, independentemente do status vacinal e com idade ³ 65 anos ou imunocomprometidos com idade ³ 18 anos","Secretaria de Ciência, Tecnologia e Inovação e do Complexo Econômico-Industrial da Saúde em 05/03/2024",Favorável à manutenção da incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 26 a 15/01/2024,Experiêcia e opinião,Técnico e científico,Favorável à manutenção da incorporação,Relatório de recomendação Final,Manutenção da incorporação,Relatório com decisão,Portaria SCTIE/MS nº 9 - Publicada em 29/05/2024,-,-
1,14,Vacina Pfizer/BioNTech para a imunização ativa de indivíduos na faixa etária de 6 meses a 5 anos incompletos para a prevenção da Covid-19,Secretaria de Vigilância em Saúde em 18/10/2022,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 06 a 15/12/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 181 - Publicada em 29/12/2022,-,-
2,13,Molnupiravir para pacientes infectados por SARS-CoV-2 não hospitalizados com alto risco de agravamento da doença,Secretaria Executiva em 16/05/2022,Desfavorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 27/06/2022 a 18/07/2022,Experiêcia e opinião,Técnico e científico,Desfavorável à incorporação,Relatório de recomendação Final,Não incorporação,Relatório com decisão,Portaria SCTIE/MS nº 102 - Publicada em 12/09/2022,-,-
3,12,Tocilizumabe para tratamento de pacientes adultos com Covid-19 hospitalizados,Ministro da Saúde em 14/04/2022,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 13/07/2022 a 01/08/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 101 - Publicada em 12/09/2022,-,-
4,11,"Cilgavimabe + tixagevimabe para profilaxia pré-exposição à Covid-19, em indivíduos adultos e pediátricos (com 12 anos de idade ou mais, pesando pelo menos 40 kg)",Secretaria Executiva em 11/04/2022 e pela AstraZeneca em 14/04/2022,Desfavorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 08 a 27/07/2022,Experiêcia e opinião,Técnico e científico,Desfavorável à incorporação,Relatório de recomendação Final,Não incorporação,Relatório com decisão,Portaria SCTIE/MS nº 115 - Publicada em 29/09/2022,-,-
5,10,Nirmatrelvir + ritonavir para o tratamento da Covid-19 em adultos que não requerem oxigênio suplementar e que apresentam risco aumentado de progressão para Covid-19 grave,Secretaria Executiva em 05/04/2022,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 18 a 27/04/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 44 - Publicada em 06/05/2022,-,-
6,9,"Baricitinibe para tratamento da Covid-19 em pacientes adultos, hospitalizados e que necessitam de oxigênio por máscara ou cateter nasal, ou que necessitam de alto fluxo de oxigênio ou ventilação não invasiva",Elli Lilly do Brasil Ltda. em 24/01/2022,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 15/03/2022 a 24/03/2022,Experiêcia e opinião,Técnico 

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [ ]:
assert "df" in globals(), "A variável df não foi encontrada."
assert "Dashboard" in globals(), "Execute primeiro as classes do projeto."


COL_DECISAO = "Decisão do Ministério da Saúde"

if COL_DECISAO not in df.columns:
    raise ValueError(
        f"Coluna não encontrada. Colunas disponíveis: {df.columns.tolist()}"
    )

opcoes = ["Todas"] + sorted(
    df[COL_DECISAO]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


filtro_decisao = widgets.Dropdown(
    options=opcoes,
    value="Todas",
    description="Decisão:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)

botao_aplicar = widgets.Button(
    description="APLICAR FILTRO",
    button_style="primary",
    icon="filter",
    layout=widgets.Layout(width="200px", height="40px")
)

botao_limpar = widgets.Button(
    description="LIMPAR FILTRO",
    button_style="",
    icon="refresh",
    layout=widgets.Layout(width="200px", height="40px")
)

resultado_painel = widgets.Output()


def atualizar_painel():

    with resultado_painel:
        clear_output(wait=True)

        try:
            if filtro_decisao.value == "Todas":
                dados_filtrados = df.copy()
            else:
                dados_filtrados = df[
                    df[COL_DECISAO].astype(str)
                    == filtro_decisao.value
                ].copy()

            print(
                "Quantidade de registros encontrados:",
                len(dados_filtrados)
            )

            if dados_filtrados.empty:
                print("Nenhum registro encontrado.")
                return


            painel_filtrado = Dashboard(dados_filtrados)


            painel_filtrado.executar()

        except Exception as erro:
            print("ERRO AO ATUALIZAR O VISUALIZADOR:")
            print(type(erro).__name__, ":", erro)

            import traceback
            traceback.print_exc()


def clicar_aplicar(botao):
    atualizar_painel()


def clicar_limpar(botao):
    filtro_decisao.value = "Todas"
    atualizar_painel()

botao_aplicar.on_click(clicar_aplicar)
botao_limpar.on_click(clicar_limpar)


display(HTML("""
<div style="
    background:#05a8e8;
    color:white;
    padding:20px;
    border-radius:12px;
    margin-top:20px;
">
<h2>PAINEL DE FILTROS</h2>
<p>Selecione uma decisão e atualize as visualizações.</p>
</div>
"""))

display(filtro_decisao)

display(
    widgets.HBox([
        botao_aplicar,
        botao_limpar
    ])
)

display(resultado_painel)

atualizar_painel()

Dropdown(description='Decisão:', layout=Layout(width='95%'), options=('Todas', '-', 'Aprovação', 'Aprovação co…

Output()